In [30]:
from datasets import load_dataset
from tqdm import tqdm
import sys
from pathlib import Path
import json

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
MODELS_PATH = PROJECT_ROOT / "models"

sys.path.append(str(SRC_PATH))

from data.dataset import CatsDogsDataset
from data.transforms import get_train_transforms

## **Limpieza del dataset**

In [31]:
dataset = load_dataset("microsoft/cats_vs_dogs", split="train")

def is_valid(example):
    try:
        example["image"].convert("RGB")
        return True
    except Exception:
        return False

clean_dataset = dataset.filter(is_valid)

print(f"Dataset original: {len(dataset)} imágenes")
print(f"Dataset limpio: {len(clean_dataset)} imágenes")

Dataset original: 23410 imágenes
Dataset limpio: 23410 imágenes


## **División del dataset**

In [32]:
splits = clean_dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="labels")

train = splits["train"]
validation = splits["test"]

## **Dataset para pytorch desacoplado de la fuente original**

In [33]:
train_pytorch_dataset = CatsDogsDataset(train, transform=get_train_transforms())

img, label = train_pytorch_dataset[0]
img.shape, label

(torch.Size([3, 224, 224]), 1)

## **Archivo json con las categorías del dataset mapeadas**

In [34]:
label_names = clean_dataset.features["labels"].names
JSON_PATH = MODELS_PATH / "label_mapping.json"

with open(str(JSON_PATH), "w") as f:
    json.dump({"labels":label_names}, f)